# momentum-buffer-update — faded example 2: Momentum buffer with gradient dampening: b = mu*b + (1-damp)*g

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `momentum-buffer-update`. Running the beacon reports progress on the `Optimizer: Momentum buffer` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Momentum buffer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`momentum-buffer-update`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "momentum-buffer-update"
DD_SUBTOPIC = "Optimizer: Momentum buffer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

PyTorch's SGD implementation optionally applies gradient dampening, which scales the current gradient contribution by `(1 - dampening)` before adding it to the buffer: `b ← μ·b + (1 - damp)·g`. Without dampening (`damp=0`) this reduces to the classical `b ← μ·b + g`. The in-place update rule is unchanged — the buffer must still be mutated, not rebound.

## Faded exercise 2

Implement `momentum_damped_step(buffer_list, grad_list, mu, dampening)` that applies:
```
b ← mu * b + (1 - dampening) * g
```
in-place for each (buffer, grad) pair. Return the list of updated buffers.

Your task: **fill in the in-place update using the dampening formula**.

**Fill in:** The in-place buffer update b.copy_(mu * b + (1 - dampening) * g) and appending b to out.

In [ ]:
import torch

def momentum_damped_step(buffer_list, grad_list, mu, dampening):
    out = []
    for b, g in zip(buffer_list, grad_list):
        raise NotImplementedError()  # TODO: The in-place buffer update b.copy_(mu * b + (1 - dampening) * g) and appending b to out.
    return out

def _test():
    import torch
    g = torch.tensor([1.0, -2.0, 3.0])
    buf = torch.zeros(3)
    mu, damp = 0.9, 0.1
    out = momentum_damped_step([buf], [g], mu, damp)
    expected = (1 - damp) * g  # first step: mu*0 + (1-damp)*g
    assert torch.allclose(buf, expected, atol=1e-6), f"got {buf}, expected {expected}"
    # Second step: b = mu*b + (1-damp)*g
    out2 = momentum_damped_step([buf], [g], mu, damp)
    expected2 = mu * expected + (1 - damp) * g
    assert torch.allclose(buf, expected2, atol=1e-6)
    # With dampening=0 reduces to classical mu*b + g
    buf2 = torch.zeros(3)
    momentum_damped_step([buf2], [g], mu, 0.0)
    assert torch.allclose(buf2, g, atol=1e-6)


def _test():
    import torch
    g = torch.tensor([1.0, -2.0, 3.0])
    buf = torch.zeros(3)
    mu, damp = 0.9, 0.1
    momentum_damped_step([buf], [g], mu, damp)
    expected = (1 - damp) * g
    assert torch.allclose(buf, expected, atol=1e-6), f"got {buf}"
    momentum_damped_step([buf], [g], mu, damp)
    expected2 = mu * expected + (1 - damp) * g
    assert torch.allclose(buf, expected2, atol=1e-6), f"got {buf}"
    # damp=0 reduces to classical
    buf2 = torch.zeros(3)
    momentum_damped_step([buf2], [g], mu, 0.0)
    assert torch.allclose(buf2, g, atol=1e-6)
    # return value should be the same buffer objects
    buf3 = torch.zeros(3)
    ret = momentum_damped_step([buf3], [g], 0.9, 0.0)
    assert ret[0] is buf3


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch

def momentum_damped_step(buffer_list, grad_list, mu, dampening):
    out = []
    for b, g in zip(buffer_list, grad_list):
        b.copy_(mu * b + (1 - dampening) * g)
        out.append(b)
    return out

def _test():
    import torch
    g = torch.tensor([1.0, -2.0, 3.0])
    buf = torch.zeros(3)
    mu, damp = 0.9, 0.1
    out = momentum_damped_step([buf], [g], mu, damp)
    expected = (1 - damp) * g
    assert torch.allclose(buf, expected, atol=1e-6)
    out2 = momentum_damped_step([buf], [g], mu, damp)
    expected2 = mu * expected + (1 - damp) * g
    assert torch.allclose(buf, expected2, atol=1e-6)
    buf2 = torch.zeros(3)
    momentum_damped_step([buf2], [g], mu, 0.0)
    assert torch.allclose(buf2, g, atol=1e-6)
```
</details>